In [1]:
from dataclasses import dataclass, field
from typing import Dict, List, Set, Optional
from transformers import AutoTokenizer, AutoModelForCausalLM
from utils.system_prompts import SYSTEM_PROMPT_CONSTR_GEN

model_id = 'meta-llama/Llama-3.1-8B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", torch_dtype="auto",
)

@dataclass
class TokTrieNode:
    """
    Node in a byte-level token trie.

    Contains
    --------
    - children: Dict[int, int] -> key: byte (int), value: index of child node in TokTrie.nodes
    - token_ids: List[int] -> list of token IDs whose byte string matches the path to this node
    """
    children: Dict[int, int] = field(default_factory=dict)
    token_ids: List[int] = field(default_factory=list)

class TokTrie:
    """
    Byte-level token trie inspired by llguidance toktrie.

    Each path from root to a terminal stores token IDs whose decoded byte string
    matches that exact path. This enables fast lookup of all tokens that are a prefix of a target byte suffix.
    """
    def __init__(self):
        """
        Initialize an empty trie.

        Contains
        --------
        - nodes: List[TokTrieNode] -> list of trie nodes; index 0 is the root
        - token_id_to_bytes: Dict[int, bytes] -> mapping from token ID to its byte string
        """
        self.nodes: List[TokTrieNode] = [TokTrieNode()]
        self.token_id_to_bytes: Dict[int, bytes] = {}

    def insert(self, token_bytes: bytes, token_id: int) -> None:
        """
        Insert a token into the trie.

        Args
        ---
        - token_bytes: bytes -> the byte string of the token to insert
        - token_id: int -> the token ID corresponding to token_bytes
        """
        node_idx = 0 # root index
        self.token_id_to_bytes[token_id] = token_bytes

        # Traverse the trie according to the bytes in token_bytes, creating new nodes as needed
        for b in token_bytes:
            child_idx = self.nodes[node_idx].children.get(b, None)
            if child_idx is None:
                # If no child for byte b, create a new node and link it
                child_idx = len(self.nodes) # new node index
                self.nodes[node_idx].children[b] = child_idx # link from parent to child
                self.nodes.append(TokTrieNode()) # add new node to the trie
            node_idx = child_idx # move to child node

        # at the terminal node -> store the token ID
        self.nodes[node_idx].token_ids.append(token_id)

    def prefix_search(self, remaining_bytes: bytes) -> Set[int]:
        """
        Find all token IDs in the trie that are a prefix of the given byte suffix.

        Args
        ----
        - remaining_bytes: bytes -> the byte string suffix for which we want to find prefix token IDs

        Example
        --------
        - At input we have remaining_bytes = b'Input'.
        - We start at the root and follow the path for bytes corresponding to 'I', 'n', 'p', 'u', 't'.
        - At each node along the path, we collect any token IDs stored at that node.
        - As a result, we find all token IDs whose byte string is a prefix of b'Input', such as tokens for 'I', 'In', and 'Input'.
        """
        node_idx = 0 # root index
        out: Set[int] = set() # to store token IDs that are a prefix of remaining_bytes

        # Traverse the trie according to the bytes in remaining_bytes, collecting token IDs along the path
        for b in remaining_bytes:
            child_idx = self.nodes[node_idx].children.get(b)
            if child_idx is None:
                # This should not happen in practice since we only search for prefixes that exist in the trie,
                # but we add this check for safety to avoid KeyError.
                break

            node_idx = child_idx # move to child node
            if self.nodes[node_idx].token_ids:
                # Update the output set with token IDs stored at this node, since they are a prefix of remaining_bytes
                out.update(self.nodes[node_idx].token_ids)

        return out

def build_toktrie_from_tokenizer(tokenizer: AutoTokenizer) -> TokTrie:
    """
    Build a toktrie structure from the given tokenizer and his vocabulary

    Args
    ----
    - tokenizer: AutoTokenizer -> the tokenizer from which to build the toktrie
    """
    toktrie = TokTrie()

    vocab = tokenizer.get_vocab()

    # For each token in the tokenizer's vocabulary, convert it to its byte string and insert it into the trie.
    for token, token_id in vocab.items():
        try:
            # surface = tokenizer.convert_tokens_to_string([token]) # convert token to surface form (string)
            surface = tokenizer.decode([token_id], clean_up_tokenization_spaces=False) # decode token ID to surface form (string)
        except Exception:
            # Fallback for tokenizers that may fail conversion on some special tokens.
            continue

        # Encode the surface string to bytes using UTF-8 encoding, which is the standard encoding for tokenizers.
        # This is done because the different tokens with or without 
        token_bytes = surface.encode("utf-8")
        toktrie.insert(token_bytes, token_id)

    return toktrie


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [2]:
toktrie = build_toktrie_from_tokenizer(tokenizer)
print(tokenizer.convert_ids_to_tokens(toktrie.prefix_search(b"Input")))

['I', 'In', 'Input']


In [3]:
# Example usage of TokTrie
# input_text = """Radek was born in Prague.     He lovers Pizza!

# He is 23        years     old.
# """
input_text = "How can you get away with such a potty mouth, i cant say $#it, dp mods are pathetic 😅"
tokenized_input = tokenizer(input_text, add_special_tokens=False)
print(f"Input text: {tokenizer.convert_ids_to_tokens(tokenized_input['input_ids'])}")
print("Prefix search results")
print("======================")
for token_id in tokenized_input['input_ids']:
    token_bytes = toktrie.token_id_to_bytes[token_id]
    print("Token:", tokenizer.convert_ids_to_tokens([token_id])[0])
    print("Prefix search results for bytes:", token_bytes)
    print(tokenizer.convert_ids_to_tokens(toktrie.prefix_search(token_bytes)))
    print()

Input text: ['How', 'Ġcan', 'Ġyou', 'Ġget', 'Ġaway', 'Ġwith', 'Ġsuch', 'Ġa', 'Ġp', 'otty', 'Ġmouth', ',', 'Ġi', 'Ġcant', 'Ġsay', 'Ġ$#', 'it', ',', 'Ġdp', 'Ġmods', 'Ġare', 'Ġpathetic', 'ĠðŁĺ', 'ħ']
Prefix search results
Token: How
Prefix search results for bytes: b'How'
['How', 'Ho', 'H']

Token: Ġcan
Prefix search results for bytes: b' can'
['Ġc', 'Ġcan', 'Ġca', 'Ġ']

Token: Ġyou
Prefix search results for bytes: b' you'
['Ġyo', 'Ġy', 'Ġ', 'Ġyou']

Token: Ġget
Prefix search results for bytes: b' get'
['Ġget', 'Ġge', 'Ġ', 'Ġg']

Token: Ġaway
Prefix search results for bytes: b' away'
['Ġa', 'Ġaway', 'Ġ', 'Ġaw']

Token: Ġwith
Prefix search results for bytes: b' with'
['Ġwi', 'Ġw', 'Ġwit', 'Ġwith', 'Ġ']

Token: Ġsuch
Prefix search results for bytes: b' such'
['Ġsuc', 'Ġs', 'Ġsuch', 'Ġ', 'Ġsu']

Token: Ġa
Prefix search results for bytes: b' a'
['Ġa', 'Ġ']

Token: Ġp
Prefix search results for bytes: b' p'
['Ġp', 'Ġ']

Token: otty
Prefix search results for bytes: b'otty'
['ot', 'otty', 'o', 'o

In [36]:
from transformers import LogitsProcessor, AutoTokenizer
from typing import Optional, Set
import torch
from utils.TokTrie import TokTrie, build_toktrie_from_tokenizer

class TrieSpanConstrainedProcessor(LogitsProcessor):
    """
    Constrained generation processor for span classification with generative LLMs.

    Behavior
    --------
    1. Outside tags, generation must copy the input text exactly (byte-wise) using the token trie
    to find all possible prefix tokens for the remaining byte suffix of the now generated token.
    2. The model may open a span and emit <SPAN><LABEL>...</LABEL>...</SPAN>.
    3. Text inside the spans is still constrained to copy the input text exactly.

    This gives probabilistic token choices while guaranteeing that removing tags reconstructs the 
    original input text exactly.
    """
    def __init__(self, labels: list[str],  input_text: str, tokenizer: AutoTokenizer,
                 toktrie: Optional[TokTrie] = None):
        """
        Initialize the processor.
        
        Args
        ----
        - labels: list[str] -> list of all possible span labels, e.g. ["PER", "LOC", "ORG"] for NER
        - input_text: str -> the input text to be classified; used for building the token trie constraints
        - tokenizer: AutoTokenizer -> the tokenizer corresponding to the model being used; needed to build the token trie
        - toktrie: TokTrie -> pre-built token trie for the tokenizer; if not provided, it will be built from the tokenizer
        """
        # Store the labels for constructing the control tokens for opening spans.
        self.labels = labels


        # Store the tokenizer and token trie for constraint logic.
        self.tokenizer = tokenizer
        self.toktrie = toktrie if toktrie is not None else build_toktrie_from_tokenizer(tokenizer)

        # Store the input text and its byte representation for tracking how much of the input has been copied so far.
        self.input_text = input_text
        self.input_tokens = tokenizer.encode(input_text, add_special_tokens=False)
        # Some input bytes may not be in the vocab, so we need to first encode the input text to tokens,
        # then convert those tokens back to bytes and concatenate to get the full byte string of the input text.
        self.input_bytes = b""
        for token_id in self.input_tokens:
            token_bytes = self.toktrie.token_id_to_bytes.get(token_id)
            if token_bytes is not None:
                self.input_bytes += token_bytes
        # self.input_bytes = input_text.encode("utf-8")
        self.input_pos = 0 # to track the current byte position in the input text
        
        # Runtime generation bookkeeping.
        self.STATE = "OUTSIDE"
        self.seq_pos = 0 # to track which token in the current control block should come next
        self.prev_len = 0 # track the len of the generated seq

        # Tracks whether at least one copy token was emitted in the current span body.
        self.span_text_has_content = False

        # Structural tokens for the constrained generation format
        self.SPAN_CLOSE = self.tokenizer.encode("</SPAN>", add_special_tokens=False)

        # Pre-encode the label tokens for quick access during generation (with and without space)
        self.label_open_blocks = {
            label: tokenizer.encode(f" <SPAN><LABEL>{label}</LABEL>", add_special_tokens=False)
            for label in labels
        }
        self.label_open_blocks_nospace = {
            label: tokenizer.encode(f"<SPAN><LABEL>{label}</LABEL>", add_special_tokens=False)
            for label in labels
        }
        self.selected_label = None # to track which label block we are currently generating
        self._active_blocks = self.label_open_blocks # to track which set of label blocks we are generating (with or without space)

        # Set of token IDs that can end the generation (e.g. EOS tokens)
        self.eos_token_ids: Set[int] = set()
        if self.tokenizer.eos_token_id is not None:
            self.eos_token_ids.add(self.tokenizer.eos_token_id)
        for tok in ["<end_of_turn>", "<|im_end|>", "<|eot_id|>"]:
            tok_id = tokenizer.convert_tokens_to_ids(tok)
            if tok_id is not None and tok_id != tokenizer.unk_token_id:
                self.eos_token_ids.add(tok_id)

    def reset(self):
        """"
        Reset the processor state for a new generation sequence.
        """
        self.STATE = "OUTSIDE"
        self.seq_pos = 0
        self.input_pos = 0
        self.selected_label = None
        self.span_text_has_content = False
        self.prev_len = 0
        self._active_blocks = None

    def _mask_except(self, scores: torch.FloatTensor, allowed_tokens: Set[int]) -> torch.FloatTensor:
        """
        Mask the scores to only allow the specified token IDs in allowed_tokens, setting all other token scores to -inf.
        """
        if not allowed_tokens:
            # Avoid all -inf rows, which break sampling (nan/inf probabilities).
            return scores
        mask = torch.ones_like(scores, dtype=torch.bool)
        mask[:, list(allowed_tokens)] = False
        scores = scores.masked_fill(mask, -float("inf"))
        return scores
    
    def _remaining_bytes(self) -> bytes:
        """
        Get the remaining byte suffix of the input text that has not been copied yet, 
        starting from the current input position.
        """
        return self.input_bytes[self.input_pos:]
    
    def _allowed_copy_tokens(self) -> Set[int]:
        """
        Get the set of token IDs that can be emitted to copy the next part of the input text, 
        based on the remaining byte suffix and the token trie.
        """
        return self.toktrie.prefix_search(self._remaining_bytes())

    def _prefer_literal_angle_bracket(self) -> bool:
        """
        If the source text currently starts with '<', prevent starting a control tag at this step.
        This avoids confusing literal '<' in text with control token '<SPAN>'.
        """
        return self._remaining_bytes().startswith(b"<")
    
    def _all_input_consumed(self) -> bool:
        """
        Check if all input bytes have been consumed (i.e. copied) so far, based on the current input position.
        """
        return self.input_pos >= len(self.input_bytes)
    
    def _consume_copy_token(self, token_id: int) -> bool:
        """
        Consume a copy token and advance the input position 
        if the given token ID corresponds to a token that can copy the next part of the input text.
        """
        if self._all_input_consumed():
            return False
        token_bytes = self.toktrie.token_id_to_bytes.get(token_id)
        if not token_bytes:
            return False
        if self._remaining_bytes().startswith(token_bytes):
            self.input_pos += len(token_bytes)
            return True
        return False
    
    def _advance_state(self, token_id: int) -> None:
        """
        Advance the FSM state based on the emitted token ID, updating the current state, sequence position, selected label, 
        and span text content flag as needed according to the constrained generation logic.
        """
        if self.STATE == "OUTSIDE":
            # First check if the last emitted token is a copy token, and if so, consume it and advance the input position accordingly.
            if self._consume_copy_token(token_id):
                return
            # Enter atomic tag block once any block-start token is emitted
            space_match = any(block and token_id == block[0] for block in self.label_open_blocks.values())
            nospace_match = any(block and token_id == block[0] for block in self.label_open_blocks_nospace.values())
            if space_match:
                if self._remaining_bytes().startswith(b" "):
                    self.input_pos += 1
                else:
                    print(f"Warning: space-prefixed block chosen but no space at input_pos {self.input_pos}")
                self.STATE = "TAG_BLOCK"
                self.seq_pos = 1
                self.selected_label = None
                self._active_blocks = self.label_open_blocks
                return
            if nospace_match:
                self.STATE = "TAG_BLOCK"
                self.seq_pos = 1
                self.selected_label = None
                self._active_blocks = self.label_open_blocks_nospace
                return
            return

        if self.STATE == "TAG_BLOCK":
            # If the last emitted token matches exactly one token in all label blocks, we know that this block must be
            # extented to the end of the block.
            if self.selected_label is None:
                matching_blocks = {
                    label: block for label, block in self._active_blocks.items() 
                    if block and token_id == block[self.seq_pos] and self.seq_pos < len(block)
                }
                if len(matching_blocks) == 1:
                    # Now we know which label block we are generating
                    self.selected_label = next(iter(matching_blocks.keys()))
                    self.seq_pos += 1
                    return
                else:
                    # More matching blocks, so just advance the seq position
                    self.seq_pos += 1
            else:
                # Should happen always since we only allow the next token in the selected block, but just in case we add this check
                if token_id == self._active_blocks[self.selected_label][self.seq_pos]:
                    self.seq_pos += 1
                    if self.seq_pos == len(self._active_blocks[self.selected_label]):
                        # We have reached the end of the selected block, so we start generating the span text
                        self.STATE = "SPAN_TEXT"
                        self.selected_label = None
                        self.seq_pos = 0
                        self.span_text_has_content = False # Spans must copy at least one token of content
                    return
                else:
                    # This should not happen in practice since we only allow the next token in the selected block,
                    # but we add this check for safety to avoid index errors.
                    print(f"Warning: {token_id} does not match expected token in selected block {self.selected_label} at position {self.seq_pos}")
                    return
        
        if self.STATE == "SPAN_TEXT":
            # First check if the last emitted token is a copy token, and if so, consume it and advance the input position accordingly.
            if self._consume_copy_token(token_id):
                self.span_text_has_content = True
                return
            # If the emitted token is not a copy token, it can only be the start of the closing tag
            if token_id == self.SPAN_CLOSE[0]:
                self.STATE = "SPAN_CLOSE"
                self.seq_pos = 1
                return
            return
        
        if self.STATE == "SPAN_CLOSE":
            # Advance through the span close sequence until it is complete, then return to OUTSIDE state
            if token_id == self.SPAN_CLOSE[self.seq_pos]:
                self.seq_pos += 1
                if self.seq_pos == len(self.SPAN_CLOSE):
                    self.STATE = "OUTSIDE"
                    self.seq_pos = 0
                    self.span_text_has_content = False
                return
            return

    def _allowed_tokens(self) -> Set[int]:
        """
        Get the set of allowed token IDs for the next generation step based on the current FSM state and seq position, 
        using the token trie to find valid copy tokens for the remaining input bytes, and allowing the appropriate special tokens
        """
        if self.STATE == "OUTSIDE":
            # Allow all tokens that can copy the next part of the input text
            allowed = self._allowed_copy_tokens()
            # Additionally, allow the tokens that can start any of the label blocks, unless the next part of the input text starts with '<'
            if not self._prefer_literal_angle_bracket():
                if self._remaining_bytes().startswith(b" "):
                    allowed.update(tok[0] for tok in self.label_open_blocks.values())
                    self._active_blocks = self.label_open_blocks # pre-set the active blocks so _advance_state sees the correct variant
                else:
                    allowed.update(tok[0] for tok in self.label_open_blocks_nospace.values())
                    self._active_blocks = self.label_open_blocks_nospace # pre-set the active blocks so _advance_state sees the correct variant
            # If all input has been consumed, allow only EOS tokens to end the generation.
            if self._all_input_consumed():
                allowed = self.eos_token_ids
            return allowed
        
        if self.STATE == "TAG_BLOCK":
            allowed = set()
            # If we have not yet disambiguated which label block we are generating, we allow any token that can be the next token in any of the label blocks
            if self.selected_label is None:
                for block in self._active_blocks.values():
                    if block and self.seq_pos < len(block) and block[self.seq_pos] not in allowed:
                        allowed.add(block[self.seq_pos])
            else:
                # Now we know which label block we are generating, so only allow the next token in that specific block.
                if self.seq_pos < len(self._active_blocks[self.selected_label]):
                    allowed.add(self._active_blocks[self.selected_label][self.seq_pos])
            return allowed
        
        if self.STATE == "SPAN_TEXT":
            # We allow all tokens that can copy the next part of the input text
            allowed = self._allowed_copy_tokens()
            if self.span_text_has_content:
                # Span close at index 0 is '</', which is a very specific token
                allowed.add(self.SPAN_CLOSE[0])
            return allowed
        
        if self.STATE == "SPAN_CLOSE":
            # Just continue the sequence
            return {self.SPAN_CLOSE[self.seq_pos]}
        return set()

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        """
        Apply the constrained generation logic to the scores.
        """
        # Get the last generated token ID from input_ids and advance the FSM state
        last_token_id = int(input_ids[0, -1])
        curr_len = input_ids.shape[1]
        if self.prev_len > 0 and curr_len > self.prev_len:
            # Only advance the state if a new token has been generated (i.e. input_ids has increased in length)
            self._advance_state(last_token_id)
        self.prev_len = curr_len

        allowed_tokens = self._allowed_tokens()
        if not allowed_tokens:
            # Dead-end in FSM/tokenization alignment: terminate safely instead of crashing sampling.
            if self.eos_token_ids:
                allowed_tokens = set(self.eos_token_ids)
        scores = self._mask_except(scores, allowed_tokens)
        
        return scores


In [37]:
labels = ["PER", "LOC", "ORG", "MISC"]

# input_text = """Radek was born in Prague. His favorite book is \"The Lord of the Rings\". 
# He is 24 years old. He loves Python programming and machine learning. 
# He works at a tech company in New York City. In his free time, he enjoys hiking and traveling to new places.
# He has a dog named Rex, which is a golden retriever breed and is very friendly and energetic. 
# Radek adopted Rex from a local animal shelter when he was a puppy, and they have been inseparable ever since. 
# Radek often takes Rex to the park for long walks and playtime, and they both enjoy spending time outdoors together."""
# input_text = "Radek was born in Prague. His favorite book is \"The Lord of the Rings\". He is 24 years old. He loves Python programming and machine learning."
# input_text = "6. Jeremie Collomb-Patton ( from France ) 23.87"
# input_text = "World Cup"
# input_text = "Radek was born in Prague. He has a dog named Rex."
# input_text = """Radek  

# was born in Prague.
# """
input_text = """i cant say $#it, dp mods are pathetic 😅"""

toktrie = build_toktrie_from_tokenizer(tokenizer)

processor = TrieSpanConstrainedProcessor(labels, input_text, tokenizer, toktrie)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT_CONSTR_GEN},
    {"role": "user", "content": input_text}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

outputs_constrained = model.generate(
    **inputs,
    max_new_tokens=516,
    do_sample=False,
    temperature=0.2,
    logits_processor=[processor],
)

output_ids_constrained = outputs_constrained[0][len(inputs.input_ids[0]):].tolist()
print("Constrained generation:")
print(tokenizer.decode(output_ids_constrained, skip_special_tokens=True, clean_up_tokenization_spaces=False))

outputs_unconstrained = model.generate(
    **inputs,
    max_new_tokens=516,
    do_sample=False,
    temperature=0.2,
)

output_ids_unconstrained = outputs_unconstrained[0][len(inputs.input_ids[0]):].tolist()
print("Unconstrained generation:")
print(tokenizer.decode(output_ids_unconstrained, skip_special_tokens=True, clean_up_tokenization_spaces=False))


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Constrained generation:
<SPAN><LABEL>MISC</LABEL>i</SPAN> can<SPAN><LABEL>MISC</LABEL>t</SPAN> <SPAN><LABEL>MISC</LABEL>say</SPAN> <SPAN><LABEL>MISC</LABEL>$</SPAN><SPAN><LABEL>MISC</LABEL>#</SPAN>it<SPAN><LABEL>MISC</LABEL>, dp</SPAN> <SPAN><LABEL>MISC</LABEL>mods</SPAN> <SPAN><LABEL>MISC</LABEL>are</SPAN> <SPAN><LABEL>MISC</LABEL>pathetic</SPAN> <SPAN><LABEL>MISC</LABEL>😅</SPAN>
Unconstrained generation:
<SPAN><LABEL>MISC</LABEL>shit</SPAN>


In [17]:
from transformers import LogitsProcessor, AutoTokenizer
from typing import List, Set, Optional
import torch
from utils.TokTrie import TokTrie, build_toktrie_from_tokenizer

class TrieSpanConstrainedProcessorTokenAware(LogitsProcessor):
    """
    Token-aware constrained generation processor for span classification with generative LLMs.

    Copy behavior:
    -------------
    1. Constrained copy is performed inside each tokenized token (not over full input bytes).
    2. For the current input token remainder (e.g. b"ade"), allowed copy tokens are all trie prefixes
       (e.g. b"a", b"ad", b"ade").
    3. If model emits b"a", we stay in the same input token and continue with b"de".

    Tagging behavior:
    -----------------
    Model may emit <SPAN><LABEL>...</LABEL> ... </SPAN> while preserving exact token-wise copying.

    """

    def __init__(self, labels: list[str],  input_text: str, tokenizer: AutoTokenizer,
                 toktrie: Optional[TokTrie] = None):
        # Store the labels for constructing the control tokens for opening spans.
        self.labels = labels

        # Store the tokenizer and token trie for constraint logic.
        self.tokenizer = tokenizer
        self.toktrie = toktrie if toktrie is not None else build_toktrie_from_tokenizer(tokenizer)

        # Store the input text and its token-level representation for tracking how much of the input has been copied so far.
        self.input_text = input_text

        # Token-level copy source: we track progress as (token index, byte offset inside that token).
        self.input_token_ids = tokenizer.encode(input_text, add_special_tokens=False)
        self.input_token_bytes: List[bytes] = [
            self.toktrie.token_id_to_bytes[tok_id]
            for tok_id in self.input_token_ids
        ]
        # Pointers to track how much of the input has been copied so far at the token level.
        self.input_token_ptr = 0
        self.input_token_byte_ptr = 0

        # Runtime generation bookkeeping.
        self.STATE = "OUTSIDE"
        self.seq_pos = 0  # used only for SPAN_CLOSE sequencing
        self.prev_len = 0 # track the len of the generated seq

        # Per-label position tracking for TAG_BLOCK disambiguation.
        # Maps label -> current position within that label's open block token sequence.
        # Blocks whose token doesn't match the emitted token are dropped immediately,
        # preventing tokens from eliminated blocks from polluting the allowed set.
        self.live_blocks: Optional[dict] = None

        # Tracks whether at least one copy token was emitted in the current span body.
        self.span_text_has_content = False

        # Structural token sequences.
        self.SPAN_CLOSE = self.tokenizer.encode("</SPAN>", add_special_tokens=False)

        # Pre-encode the label tokens for quick access during generation (with and without space)
        self.label_open_blocks = {
            label: tokenizer.encode(f" <SPAN><LABEL>{label}</LABEL>", add_special_tokens=False)
            for label in labels
        }
        self.label_open_blocks_nospace = {
            label: tokenizer.encode(f"<SPAN><LABEL>{label}</LABEL>", add_special_tokens=False)
            for label in labels
        }
        self.selected_label = None  # set when entering SPAN_TEXT, not during TAG_BLOCK
        self._active_blocks = self.label_open_blocks  # which variant (space / no-space) is active

        # End tokens accepted once all input tokens are fully consumed.
        self.eos_token_ids: Set[int] = set()
        if tokenizer.eos_token_id is not None:
            self.eos_token_ids.add(tokenizer.eos_token_id)
        for tok in ["<end_of_turn>", "<|im_end|>", "<|eot_id|>"] :
            tok_id = tokenizer.convert_tokens_to_ids(tok)
            if tok_id is not None and tok_id != tokenizer.unk_token_id:
                self.eos_token_ids.add(tok_id)

    def reset(self):
        """
        Reset the processor state for a new generation sequence.
        """
        self.STATE = "OUTSIDE"
        self.seq_pos = 0
        self.input_token_ptr = 0
        self.input_token_byte_ptr = 0
        self.selected_label = None
        self.live_blocks = None
        self.span_text_has_content = False
        self.prev_len = 0
        self._active_blocks = None

    def _mask_except(self, scores: torch.FloatTensor, allowed_tokens: Set[int]) -> torch.FloatTensor:
        """
        Mask the scores to only allow the specified token IDs in allowed_tokens, setting all other token scores to -inf.
        """
        if not allowed_tokens:
            # Avoid all -inf rows, which break sampling (nan/inf probabilities).
            return scores
        mask = torch.ones_like(scores, dtype=torch.bool)
        mask[:, list(allowed_tokens)] = False
        scores = scores.masked_fill(mask, -float("inf"))
        return scores

    def _all_input_consumed(self) -> bool:
        """
        Check if all input tokens have been fully consumed (i.e. copied) so far, 
        based on the current token pointer and byte pointer.
        """
        return self.input_token_ptr >= len(self.input_token_bytes)

    def _normalize_token_cursor(self) -> None:
        """Advance to the next input token when current token bytes are exhausted."""
        while not self._all_input_consumed():
            cur_len = len(self.input_token_bytes[self.input_token_ptr])
            if self.input_token_byte_ptr < cur_len:
                break
            self.input_token_ptr += 1
            self.input_token_byte_ptr = 0
    
    def _current_remaining_bytes(self) -> bytes:
        """
        Get the remaining byte suffix of the current input token that has not been copied yet, 
        based on the current token pointer and byte pointer.
        """
        self._normalize_token_cursor()
        if self._all_input_consumed():
            return b""
        cur = self.input_token_bytes[self.input_token_ptr]
        return cur[self.input_token_byte_ptr:]
    
    def _allowed_copy_tokens(self) -> Set[int]:
        """
        Get the set of token IDs that can be emitted to copy the next part of the input text, 
        based on the remaining byte suffix of the current input token and the token trie.
        """
        remaining = self._current_remaining_bytes()
        if not remaining:
            return set()
        return self.toktrie.prefix_search(remaining)

    def _prefer_literal_angle_bracket(self) -> bool:
        """
        If the source text currently starts with '<', prevent starting a control tag at this step.
        This avoids confusing literal '<' in text with control token '<SPAN>'.
        """
        return self._current_remaining_bytes().startswith(b"<")

    def _consume_copy_token(self, token_id: int) -> bool:
        """
        Consume a copy token and advance the input token pointer and byte pointer 
        if the given token ID corresponds to a token that can copy the next part of the input text.
        """
        if self._all_input_consumed():
            return False
        token_bytes = self.toktrie.token_id_to_bytes.get(token_id)
        if not token_bytes:
            return False
        remaining = self._current_remaining_bytes()
        if not remaining.startswith(token_bytes):
            return False

        # Advance the byte pointer by the length of the consumed token bytes
        self.input_token_byte_ptr += len(token_bytes)
        cur_len = len(self.input_token_bytes[self.input_token_ptr])

        # If the current input token was fully consumed, move to next token.
        if self.input_token_byte_ptr >= cur_len:
            self.input_token_ptr += 1
            self.input_token_byte_ptr = 0
        return True

    def _advance_state(self, token_id: int) -> None:
        """
        Advance the FSM state based on the emitted token ID, updating the current state, sequence position, selected label,
        and span text content flag as needed according to the constrained generation logic.
        """
        if self.STATE == "OUTSIDE":
            # First check if the last emitted token is a copy token, and if so, consume it and advance the input position accordingly.
            if self._consume_copy_token(token_id):
                return
            # Enter atomic tag block once any block-start token is emitted
            space_match = any(block and token_id == block[0] for block in self.label_open_blocks.values())
            nospace_match = any(block and token_id == block[0] for block in self.label_open_blocks_nospace.values())
            if space_match:
                remaining = self._current_remaining_bytes()
                can_copy_after_space = bool(remaining[1:]) and bool(self.toktrie.prefix_search(remaining[1:]))
                if remaining.startswith(b" ") and can_copy_after_space:
                    self.input_token_byte_ptr += 1
                    self._normalize_token_cursor()
                else:
                    print(
                        f"Warning: rejected space-prefixed block at token {self.input_token_ptr}, "
                        f"byte {self.input_token_byte_ptr}; cannot continue copying after consuming leading space"
                    )
                    return
                self.STATE = "TAG_BLOCK"
                self.live_blocks = {label: 1 for label in self.labels}
                self.selected_label = None
                self._active_blocks = self.label_open_blocks
                return
            if nospace_match:
                self.STATE = "TAG_BLOCK"
                self.live_blocks = {label: 1 for label in self.labels}
                self.selected_label = None
                self._active_blocks = self.label_open_blocks_nospace
                return
            return

        if self.STATE == "TAG_BLOCK":
            # Advance each live block if its next expected token matches the emitted token,
            # and drop blocks that do not match. This prevents tokens from eliminated blocks
            # (e.g. "ISC" from the MISC block after ">" was chosen instead of ">M") from
            # appearing in the allowed set at subsequent steps.
            new_live = {}
            for label, pos in self.live_blocks.items():
                block = self._active_blocks[label]
                if pos < len(block) and block[pos] == token_id:
                    new_pos = pos + 1
                    if new_pos == len(block):
                        # This label's open block is fully emitted: transition to SPAN_TEXT.
                        self.STATE = "SPAN_TEXT"
                        self.selected_label = label
                        self.live_blocks = None
                        self.seq_pos = 0
                        self.span_text_has_content = False
                        return
                    new_live[label] = new_pos
            self.live_blocks = new_live
            return

        if self.STATE == "SPAN_TEXT":
            # First check if the last emitted token is a copy token, and if so, consume it and advance the input position accordingly.
            if self._consume_copy_token(token_id):
                self.span_text_has_content = True
                return
            # If the emitted token is not a copy token, it can only be the start of the closing tag
            if token_id == self.SPAN_CLOSE[0]:
                self.STATE = "SPAN_CLOSE"
                self.seq_pos = 1
                return
            return

        if self.STATE == "SPAN_CLOSE":
            # Advance through the span close sequence until it is complete, then return to OUTSIDE state
            if self.seq_pos < len(self.SPAN_CLOSE) and token_id == self.SPAN_CLOSE[self.seq_pos]:
                self.seq_pos += 1
                if self.seq_pos == len(self.SPAN_CLOSE):
                    self.STATE = "OUTSIDE"
                    self.seq_pos = 0
                    self.span_text_has_content = False
                return

    def _allowed_tokens(self) -> Set[int]:
        """
        Get the set of allowed token IDs for the next generation step based on the current FSM state and seq position,
        using the token trie to find valid copy tokens for the remaining input bytes, and allowing the appropriate special tokens
        """
        if self.STATE == "OUTSIDE":
            # Allow all tokens that can copy the next part of the input text based on the current token-level position
            allowed = self._allowed_copy_tokens()
            # Additionally, allow the tokens that can start any of the label blocks, unless the next part of the input text starts with '<'
            if not self._prefer_literal_angle_bracket():
                remaining = self._current_remaining_bytes()
                can_copy_after_space = bool(remaining[1:]) and bool(self.toktrie.prefix_search(remaining[1:]))
                if remaining.startswith(b" ") and can_copy_after_space:
                    allowed.update(tok[0] for tok in self.label_open_blocks.values())
                    self._active_blocks = self.label_open_blocks # pre-set the active blocks so _advance_state sees the correct variant
                else:
                    allowed.update(tok[0] for tok in self.label_open_blocks_nospace.values())
                    self._active_blocks = self.label_open_blocks_nospace # pre-set the active blocks so _advance_state sees the correct variant
            # If all input has been consumed, allow only EOS tokens to end the generation.
            if self._all_input_consumed():
                allowed = set(self.eos_token_ids)
            return allowed

        if self.STATE == "TAG_BLOCK":
            # Only allow the next token from blocks that are still live (consistent with
            # tokens emitted so far). Eliminated blocks are already absent from live_blocks.
            allowed = set()
            for label, pos in self.live_blocks.items():
                block = self._active_blocks[label]
                if pos < len(block):
                    allowed.add(block[pos])
            return allowed

        if self.STATE == "SPAN_TEXT":
            # We allow all tokens that can copy the next part of the input text
            allowed = self._allowed_copy_tokens()
            if self.span_text_has_content:
                # Span close at index 0 is '</', which is a very specific token
                allowed.add(self.SPAN_CLOSE[0])
            return allowed

        if self.STATE == "SPAN_CLOSE":
            # Just continue the span close sequence until it is complete
            return {self.SPAN_CLOSE[self.seq_pos]}

        return set()

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        """
        Apply the constrained generation logic to the scores.
        """
        # Get the last generated token ID from input_ids and advance the FSM state
        last_token_id = int(input_ids[0, -1])
        curr_len = input_ids.shape[1]
        if self.prev_len > 0 and curr_len > self.prev_len:
            self._advance_state(last_token_id)
        self.prev_len = curr_len

        allowed_tokens = self._allowed_tokens()
        if not allowed_tokens:
            # Dead-end in FSM/tokenization alignment: terminate safely instead of crashing sampling.
            if self.eos_token_ids:
                allowed_tokens = set(self.eos_token_ids)
        scores = self._mask_except(scores, allowed_tokens)
        
        return scores
    
def generate_with_trie_processor(model, tokenizer, processor, input_text, labels, system_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": input_text}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    outputs_constrained = model.generate(
        **inputs,
        max_new_tokens=516,
        do_sample=True,
        temperature=0.2,
        logits_processor=[processor],
    )

    output_ids_constrained = outputs_constrained[0][len(inputs.input_ids[0]):].tolist()
    print("Constrained generation:")
    print(tokenizer.decode(output_ids_constrained, skip_special_tokens=True, clean_up_tokenization_spaces=False))

    outputs_unconstrained = model.generate(
        **inputs,
        max_new_tokens=516,
        do_sample=True,
        temperature=0.2,
    )
    output_ids_unconstrained = outputs_unconstrained[0][len(inputs.input_ids[0]):].tolist()

    print("Unconstrained generation:")
    print(tokenizer.decode(output_ids_unconstrained, skip_special_tokens=True, clean_up_tokenization_spaces=False))

    return output_ids_constrained, output_ids_unconstrained

# Example usage for your current NER-style span-label task:
labels = ["PER", "LOC", "ORG", "MISC"]
# text = "On podium , Karin said \" I am very happy to receive this award on behalf of my team at Microsoft Research . \""
# text = "Radek was born in Prague. His favorite book is \"The Lord of the Rings\"."
# text = """Radek was born in Prague. His favorite book is \"The Lord of the Rings\". 
# He is 24 years old. He loves Python programming and machine learning. 
# He works at a tech company in New York City. In his free time, he enjoys hiking and traveling to new places.
# He has a dog named Rex, which is a golden retriever breed and is very friendly and energetic. 
# Radek adopted Rex from a local animal shelter when he was a puppy, and they have been inseparable ever since. 
# Radek often takes Rex to the park for long walks and playtime, and they both enjoy spending time outdoors together."""
# text = """Radek  

# was born in Prague."""
text = "Ajax Amsterdam 18 7 8 3 23 16 29 Results of German first division A police spokesman said two youths believed to be supporters of President Nelson Mandela 's African National Congress ( ANC ) had been killed when unknown gunmen opened fire at the rural settlement of Izingolweni on KwaZulu-Natal province 's south coast on Thursday night . CAROLINA 9 4 0 292 164 MADRID 1996-12-07"
toktrie = build_toktrie_from_tokenizer(tokenizer)
processor = TrieSpanConstrainedProcessorTokenAware(labels, text, tokenizer, toktrie)
output_ids_constrained, output_ids_unconstrained = generate_with_trie_processor(model, tokenizer, processor, text, labels, SYSTEM_PROMPT_CONSTR_GEN)
print(f"Consumed input tokens: {processor.input_token_ptr}/{len(processor.input_token_bytes)}")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Constrained generation:
<SPAN><LABEL>PER</LABEL>Ajax Amsterdam</SPAN> 18 7 8 3 23 16 29 Results of <SPAN><LABEL>MISC</LABEL>German first division A</SPAN> <SPAN><LABEL>ORG</LABEL>police spokesman</SPAN> said two youths believed to be supporters of <SPAN><LABEL>PER</LABEL>President Nelson Mandela</SPAN> 's <SPAN><LABEL>ORG</LABEL>African National Congress</SPAN> ( <SPAN><LABEL>ORG</LABEL>ANC</SPAN> ) had been killed when unknown gunmen opened fire at the rural settlement of <SPAN><LABEL>MISC</LABEL>Izingolweni</SPAN> on <SPAN><LABEL>LOC</LABEL>KwaZulu-Natal province</SPAN> 's south coast on <SPAN><LABEL>MISC</LABEL>Thursday night</SPAN> . <SPAN><LABEL>MISC</LABEL>CAROLINA</SPAN> 9 4 0 292 164 <SPAN><LABEL>MISC</LABEL>MA</SPAN><SPAN><LABEL>PER</LABEL>DRI</SPAN><SPAN><LABEL>MISC</LABEL>D</SPAN> 1996-12-07
Unconstrained generation:
<SPAN><LABEL>PER</LABEL>Ajax Amsterdam</SPAN> 18 7 8 3 23 16 29 Results of <SPAN><LABEL>MISC</LABEL>German first division A</SPAN> <SPAN><LABEL>ORG</LABEL>A pol